# 02 — Mamba-130m generation with the naive scan

Load the pretrained `state-spaces/mamba-130m-hf` checkpoint, route its mixer's selective scan through our `selective_scan_naive`, and confirm generated token IDs match the unpatched HuggingFace baseline.

This exercises three things simultaneously:
1. `weights.load_mamba_hf_checkpoint` reconstructs a HF Mamba config + state dict.
2. `weights.load_layer_into_block` loads one mixer layer into our `MambaBlock`.
3. A monkey-patch swaps HF's `MambaMixer.slow_forward` scan with ours for end-to-end generation.

## 1. Setup — device, model, tokenizer

If CUDA is available we place the model on GPU (Mamba-130m is ~500 MB in fp32).

In [1]:
import torch, types
from transformers import AutoModelForCausalLM, AutoTokenizer
from mamba_minimal.scan_naive import selective_scan_naive
from mamba_minimal.weights import (
    load_mamba_hf_checkpoint, load_layer_into_block, extract_mixer_state,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL  = 'state-spaces/mamba-130m-hf'
tok    = AutoTokenizer.from_pretrained(MODEL)
model  = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()
device, MODEL

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


('cpu', 'state-spaces/mamba-130m-hf')

## 2. Mixer-level parity on a single layer

Build a fresh `MambaBlock`, load layer-0 weights into it, run it on random input, and compare against the HF mixer's own output. Bit-level parity at fp32 is the expectation.

In [2]:
ckpt = load_mamba_hf_checkpoint(MODEL)
ours = load_layer_into_block(ckpt, layer_idx=0).to(device).eval()
hf_mixer = model.backbone.layers[0].mixer.eval()

torch.manual_seed(0)
d_model = ckpt.config.hidden_size
x = torch.randn(2, 16, d_model, device=device)
with torch.no_grad():
    y_ours = ours(x)
    y_hf   = hf_mixer(x)
print('max |ours - hf_mixer|:', (y_ours - y_hf).abs().max().item())

max |ours - hf_mixer|: 0.0


## 3. Swap HF's scan with `selective_scan_naive`

We override `MambaMixer.slow_forward` in each layer with a version that performs the same projections and conv, then delegates the scan to our naive implementation. Setting `use_cache=False` forces the full-sequence slow path for every decode step, so every generated token exercises our code.

In [3]:
def naive_slow_forward(self, input_states, cache_params=None,
                       cache_position=None, attention_mask=None):
    # Mirrors transformers.models.mamba.modeling_mamba.MambaMixer.slow_forward,
    # but swaps the scan body with selective_scan_naive.
    batch_size, seq_len, _ = input_states.shape
    dtype = input_states.dtype

    projected_states = self.in_proj(input_states).transpose(1, 2)
    hidden_states, gate = projected_states.chunk(2, dim=1)

    if attention_mask is not None:
        hidden_states = hidden_states * attention_mask.unsqueeze(1)

    conv_weight = self.conv1d.weight.view(self.conv1d.weight.size(0),
                                         self.conv1d.weight.size(2))
    hidden_states = self.act(
        self.conv1d(hidden_states)[..., :seq_len]
    )
    if attention_mask is not None:
        hidden_states = hidden_states * attention_mask.unsqueeze(1)

    ssm_parameters = self.x_proj(hidden_states.transpose(1, 2))
    time_step, Bm, Cm = torch.split(
        ssm_parameters,
        [self.time_step_rank, self.ssm_state_size, self.ssm_state_size],
        dim=-1,
    )
    # HF applies dt_proj (Linear, bias included) and softplus BEFORE the scan.
    # So we pass delta_bias=None and delta_softplus=False to selective_scan_naive —
    # the bias and softplus are already baked into `delta`.
    discrete_time_step = torch.nn.functional.softplus(
        self.dt_proj(time_step)
    ).transpose(1, 2)

    A = -torch.exp(self.A_log.float())
    y = selective_scan_naive(
        u=hidden_states,
        delta=discrete_time_step,
        A=A,
        B=Bm.transpose(1, 2).contiguous(),
        C=Cm.transpose(1, 2).contiguous(),
        D=self.D,
        z=gate,
        delta_bias=None,
        delta_softplus=False,
    )
    return self.out_proj(y.transpose(1, 2)).to(dtype)

# Install on every layer and disable the fast CUDA path.
for layer in model.backbone.layers:
    layer.mixer.slow_forward = types.MethodType(naive_slow_forward, layer.mixer)
    layer.mixer.cuda_kernels_forward = types.MethodType(naive_slow_forward, layer.mixer)
print('patched', len(model.backbone.layers), 'layers')

patched 24 layers


## 4. Sanity — forward-pass logits match HF exactly

Reload a fresh HF model (unpatched) as the oracle and compare the logits tensor for a single prefill. The patched model replaces HF's scan — if the math is right, logits agree to within fp32 round-off.

In [4]:
oracle_model = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()
prompt = 'Mamba is useful because'
ids = tok(prompt, return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    logits_patched = model(ids, use_cache=False).logits
    logits_oracle  = oracle_model(ids, use_cache=False).logits
print('max |logits diff|:', (logits_patched - logits_oracle).abs().max().item())

max |logits diff|: 0.0


## 5. Generate text — patched naive-scan path

Greedy decoding with `use_cache=False` so every new token reruns the full sequence through our scan (inefficient, but every token is ours). Expect token-exact agreement with the oracle.

In [5]:
with torch.no_grad():
    out_patched = model.generate(ids, max_new_tokens=20, do_sample=False,
                                 use_cache=False)
    out_oracle  = oracle_model.generate(ids, max_new_tokens=20, do_sample=False,
                                       use_cache=False)
same_tokens = (out_patched == out_oracle).all().item()
print('token-exact:', same_tokens)
print('patched :', tok.decode(out_patched[0]))
print('oracle  :', tok.decode(out_oracle[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


token-exact: True
patched : Mamba is useful because it is a very good example of how to use the
<Mamba>  <Mamba
oracle  : Mamba is useful because it is a very good example of how to use the
<Mamba>  <Mamba


## Takeaway

Generation from Mamba-130m with our naive scan matches the unpatched HuggingFace reference token-for-token. Combined with the primitive-level `mamba_ssm` parity from notebook 01 and `tests/test_naive_vs_reference.py`, the naive path is a verified oracle-compatible reference.

**Next:** build a standalone `MambaModel` (embedding → N × block → RMSNorm → LM head) + a custom `generate()` that carries the SSM state, so we no longer need to ride HF's scaffolding for generation.